# 示例策略：多空概率简单预测
接下来我们将展示一个示例策略，这个策略使用一个自定义的 Kera 神经网络进行多空预测，根据预测结果进行交易决策。

要运行这个实验，请选择 conda_tensorflow2_p36 内核。

在开始之前，我们先更新一下 Tensorflow 和 Keras 至最新版本：

In [ ]:
!pip install --upgrade tensorflow keras

### 定义参数

我们首先定义一些实验需要用到的环境变量：

In [ ]:
import boto3

aws_account_id = boto3.client('sts').get_caller_identity().get('Account')
repository_name = 'long-short-predict'
aws_region = 'us-east-1'
my_name = 'xxx'
bucket = 'algo-trading-workshop-{}'.format(my_name)

### 获取数据

与之前实验的一样，我们选择一支股票，然后通过 Athena 获取行情数据：

In [ ]:
!pip install awswrangler

import awswrangler as wr

s3_output = wr.athena.create_athena_bucket()

def execute_query(database, sql):
    
    query_execution_id = wr.athena.start_query_execution(database=database, sql=sql)
    response = wr.athena.get_query_execution(query_execution_id=query_execution_id)
    wr.athena.wait_query(query_execution_id=query_execution_id)
    OutputLocation = response['ResultConfiguration']['OutputLocation']
    
    return OutputLocation

In [ ]:
database = 'stock-data'
table = 'stock_day'
fields = '*'
ticker = '600519'
orderby = 'tradedate'
sort = 'ASC'

sql = f'''
SELECT {fields}
FROM "{database}"."{table}"
WHERE ticker='{ticker}'
AND tradedate>='2015-01-01'
ORDER BY {orderby}
{sort}
'''

output_location = execute_query(database, sql)
print(output_location)

In [ ]:
df = wr.s3.read_csv(path=[output_location])

df['ticker'] = df['ticker'].apply(lambda x: str(x))
df['ticker'] = df['ticker'].apply(lambda x: '0'*(6-len(x)) + x)
df['openprice'] = df['openprice'] * df['accumadjfactor'] / df['accumadjfactor'].iloc[-1]
df['closeprice'] = df['closeprice'] * df['accumadjfactor'] / df['accumadjfactor'].iloc[-1]
df['highestprice'] = df['highestprice'] * df['accumadjfactor'] / df['accumadjfactor'].iloc[-1]
df['lowestprice'] = df['lowestprice'] * df['accumadjfactor'] / df['accumadjfactor'].iloc[-1]
df = df[df['isopen'] == True]
df.drop('isopen', 1, inplace=True)
df.drop('accumadjfactor', 1, inplace=True)
df.set_index('tradedate', inplace=True)
df.sort_index(0, inplace=True)

df.rename(columns={'openprice': 'open'}, inplace=True)
df.rename(columns={'closeprice': 'close'}, inplace=True)
df.rename(columns={'highestprice': 'high'}, inplace=True)
df.rename(columns={'lowestprice': 'low'}, inplace=True)
df.rename(columns={'turnovervol': 'volume'}, inplace=True)
df['openinterest'] = 0 # A股回测中一般并不考虑利率，通常可以直接设为 0
df.head()

In [ ]:
!mkdir -p {repository_name}/input/data
df.to_csv('{}/input/data/data_raw.csv'.format(repository_name))

### 定义超参

在开始之前我们先定义一下这个算法中包含几个重要的参数：
- long_threshold：当模型预测止盈概率超过此数值时做多 (0 到 1 之间的小数)。
- short_threshold：当模型预测止损概率超过此数值时做空 (0 到 1 之间的小数)。如果设为大于等于 1，策略就不会做空。
- profit_target：当交易盈利超过此比例时止盈（0 到 1 之间的小数）。
- stop_target：当交易损失超过此比例时止损（0 到 1 之间的小数）。
- look_back：每次预测使用过去多少个交易日（整数）的历史数据。（look_back = repeat_count * repeat_step）
- forward_window：每次预测向前推导多少个交易日（整数）。

In [ ]:
!mkdir {repository_name}

In [ ]:
params = { 
    "long_threshold" : 0.5,
    "short_threshold" : 1,
    "profit_target" : 0.02,
    "stop_target" : 0.01,
    "repeat_count": 20,
    "repeat_step": 1,
    "forward_window": 10
}

long_threshold = params['long_threshold']
short_threshold = params['short_threshold']
profit_target = params['profit_target']
stop_target = params['stop_target']

repeat_count = params['repeat_count']
repeat_step = params['repeat_step']
look_back = repeat_count * repeat_step
forward_window = params['forward_window']

# 将参数以 json 格式保存至指定目录
import json

with open('{}/hyperparameters.json'.format(repository_name), 'w') as fp:
    json.dump(params, fp)

### 模型训练

在这个演示中，我们获取了 2010 年至今的数据用于模型训练和预测。这个训练过程将运行 100 个 epoch：

In [ ]:
!pip install backtrader
!pip uninstall -y matplotlib
!pip install matplotlib==3.1.3
!pip install talib-binary

In [ ]:
from __future__ import print_function

import datetime
import math
import numpy as np
import os
import pandas as pd
import sys
import talib as ta
import traceback

from keras.layers import Dropout, Dense
from keras.wrappers.scikit_learn import KerasClassifier
from keras.models import Sequential
from keras.wrappers.scikit_learn import KerasRegressor


# Optional
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

config_path = '{}/'.format(repository_name)
input_path = '{}/input/data/data_train.csv'.format(repository_name)
raw_path = '{}/input/data/data_raw.csv'.format(repository_name)
train_path = '{}/input/data/data_train.csv'.format(repository_name)
test_path = '{}/input/data/data_test.csv'.format(repository_name)
output_path = '{}/output'.format(repository_name)
model_path = '{}/model.h5'.format(repository_name)


# Process and prepare the data
def get_data(params):
    
    # get and save hyperparameters
    long_threshold = float(params['long_threshold'])
    short_threshold = float(params['short_threshold'])
    profit_target = float(params['profit_target'])
    stop_target = float(params['stop_target'])

    repeat_count = int(params['repeat_count'])
    repeat_step = int(params['repeat_step'])
    look_back = repeat_count * repeat_step
    forward_window = int(params['forward_window'])
    
    # read data from s3
    df = pd.read_csv(raw_path, index_col=0)
    closePrice = df["close"]

    # use talib to calculate SMA and ROC
    ## set header for transformed data
    header = ["tradedate", "close"]
    for i in range(0, repeat_count):
        header.append("sma" + str((i+1) * repeat_step))
    for i in range(0, repeat_count):
        header.append("roc" + str((i+1) * repeat_step))
    header.append("long")
    header.append("short")

    data = []

    ## SMA
    inputs = {'close': np.array(closePrice)}
    sma = []
    for i in range(0, repeat_count):
        sma.append(ta.SMA(np.array(closePrice), timeperiod=(i+1) * repeat_step + 1))
    ## ROC - Rate of change : ((price/prevPrice)-1)*100
    roc = []
    for i in range(0, repeat_count):
        roc.append(ta.ROC(np.array(closePrice), timeperiod=(i+1) * repeat_step + 1))

    ## count long and short
    long_count = 0
    short_count = 0
    n_count = 0
    n = 0
    for idx in df.index:
        if n < len(df) - forward_window - 1:
            idx_0 = idx
            close_price = df.loc[idx, 'close']
            temp = []
            temp.append(idx)

            temp2 = []
            temp2.append(close_price)

            # sma
            for i in range(0, repeat_count):
                if np.isnan(sma[i][n]):
                    temp2.append(close_price)
                else:
                    temp2.append(sma[i][n])

            min_value = min(temp2)
            max_value = max(temp2)
            for i in temp2:
                if max_value == min_value:
                    temp.append(0)
                else:
                    temp.append((i - min_value) / (max_value - min_value))

            for i in range(0, repeat_count):
                if np.isnan(roc[i][n]):
                    temp.append(0)
                else:
                    temp.append(roc[i][n])

            rClose = closePrice[(n+1):min(len(df)-1, n+1+forward_window)].values.tolist()
            min_value = min(rClose)
            max_value = max(rClose)

            # long condition
            if max_value >= close_price * (1+profit_target) and min_value >= close_price * (1-stop_target):
                long_count += 1
                temp.append(1)
            else:
                temp.append(0)

            # short condition
            if min_value <= close_price * (1-stop_target) and max_value <= close_price * (1+profit_target):
                short_count += 1
                temp.append(1)
            else:
                temp.append(0)

            data.append(temp)
            n += 1

    print("long：%s, short：%s" % (long_count,short_count))
    df2 = pd.DataFrame(data, columns=header)
    df2.set_index('tradedate', inplace=True)
    print('Range:', df2.index[0], '-', df2.index[-1])
    
    # save data
    df_train = df2.iloc[:800]
    print('Training set:', df_train.index[0], '-', df_train.index[-1])
    print('Location:', train_path)
    df_train.to_csv(train_path)

    df_test = df2.iloc[800:1000]
    print('Testing set:', df_test.index[0], '-', df_test.index[-1])
    print('Location:', test_path)
    df_test.to_csv(test_path)
    

# Process and prepare the data
def data_process(df, yLen, b):
    
    dataX = []
    dataY = []
    for idx, row in df.iterrows():
        row1 = []
        r = row[1:len(row)-yLen]
        for a in r:
            row1.append(a)
        x = np.array(row1, dtype=float)
        y = np.array(row[len(row)-yLen:], dtype=float)
        b = len(x)
        dataX.append(x)
        dataY.append(y)
        
    dataX = np.array(dataX)
    dataY = np.array(dataY)
    
    return dataX, dataY, b


def build_classifier(b, yLen):
    
    print("build_classifier:b=%s,yLen=%s" % (b, yLen))
    model = Sequential()
    model.add(Dense(b, input_dim=b, kernel_initializer='normal', activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(int(b/2), kernel_initializer='normal', activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(yLen, kernel_initializer='normal', activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    return model


def generate_model(dataX, dataY, b, yLen):
    
    model = build_classifier(b, yLen)
    model.fit(dataX, dataY, epochs=100, batch_size=1)
    scores = model.evaluate(dataX, dataY, verbose=0)
    print("Training Data %s: %.2f%%" % (model.metrics_names[1], scores[1]*100))
    
    return model


def train():
    
    try:
        with open("{}/hyperparameters.json".format(config_path)) as json_file:
            params = json.load(json_file)
        print('Parameter load success')
    except Exception as e:
        print(e)

    get_data(params)
    
    print('Starting the training.')
    
    yLen = 2
    b = 0
    
    try:
        
        df = pd.read_csv(input_path)
        dataX, dataY, b = data_process(df, yLen, b)
        print('b:', b, 'yLen:', yLen)
        model = generate_model(dataX, dataY, b, yLen)
        model.save(model_path)
        
        print('Training is complete. Model saved.')
        
        df = pd.read_csv(test_path)
        dataX, dataY, b = data_process(df, yLen, b)
        print('b:', b, 'yLen:', yLen)
        scores = model.evaluate(dataX, dataY, verbose=0)
        print("Test Data %s: %.2f%%" % (model.metrics_names[1], scores[1]*100))
        
    except Exception as e:
        # Write out an error file. This will be returned as the failure
        # Reason in the DescribeTrainingJob result.
        trc = traceback.format_exc()
        with open(os.path.join(output_path, 'failure'), 'w') as s:
            s.write('Exception during training: ' + str(e) + '\n' + trc)
        # Printing this causes the exception to be in the training job logs
        print(
            'Exception during training: ' + str(e) + '\n' + trc,
            file=sys.stderr)
        # A non-zero exit code causes the training job to be marked as Failed.
        sys.exit(255)

        
if __name__ == '__main__':
    train()

    # A zero exit code causes the job to be marked a Succeeded.
    sys.exit(0)

看到 SystemExit: 0 则表明模型训练成功。

训练完成后的模型保存在 model/ 路径下，应该可以看到路径下有名为 model.h5 的模型文件生成。可以运行以下代码尝试在 TensorFlow 中加载模型：

In [ ]:
from keras.models import load_model

model_path = '{}/model.h5'.format(repository_name)
print('Model path:', model_path)
try:
    model = load_model(model_path)
    print('Model load success')
    
except Exception as e:
    print(e)

In [ ]:
model.summary()

### 运行回测

接下来我们将定义运行回测任务，看看模型在实际回测中的表现。这个回测任务将使用除了训练和测试集之外的交易日作为样本。

In [ ]:
# 回测数据
df_backtest= df.iloc[1000:]
print('Backtest set:', df_backtest.index[0], '-', df_backtest.index[-1])
df_backtest.head()

# 超参
with open("{}/hyperparameters.json".format(repository_name)) as json_file:
    params = json.load(json_file)

接下来的脚本包含了运行回测所需的策略代码。该策略将从示例的路径中加载之前训练好的模型。

In [ ]:
import backtrader as bt
import json
import math
import numpy as np
import pandas as pd
import talib as ta

import keras
from keras import backend as K
from keras.models import load_model


class MyStrategy(bt.Strategy):
    
    params = (
        ('model_path', ''), 
        ('long_threshold', 0.5), 
        ('short_threshold', 1), 
        ('profit_target', 0.02), 
        ('stop_target', 0.01), 
        ('repeat_count', 20), 
        ('repeat_step', 1), 
        ('printlog', True))

    def __init__(self):
        super(MyStrategy, self).__init__()

        self.order = None
        self.orderPlaced = False
                                
        self.model = load_model(self.params.model_path)
        
        # input / indicators
        self.long_threshold = self.params.long_threshold
        self.short_threshold = self.params.short_threshold
        self.repeat_count = self.params.repeat_count
        self.repeat_step = self.params.repeat_step
        self.profit_target = self.params.profit_target
        self.stop_target = self.params.stop_target
    
        self.sma=[]
        self.roc=[]
        for i in range(0, self.repeat_count):
            self.sma.append(bt.talib.SMA(self.data, timeperiod=(i+1)*self.repeat_step + 1, plot=False))
            self.roc.append(bt.talib.ROC(self.data, timeperiod=(i+1)*self.repeat_step + 1, plot=False))
        
    def next(self):
        super(MyStrategy, self).next()
        
        idx_0 = self.datas[0].datetime.datetime(0)
        close_price = self.datas[0].close
        temp = []
        
        temp2 = []
        temp2.append(close_price)

        ## sma
        for i in range(0, self.repeat_count):
            if math.isnan(self.sma[i][0]):
                temp2.append(close_price)
            else:
                temp2.append(self.sma[i][0])
                
        min_value = min(temp2)
        max_value = max(temp2)
        for i in temp2:
            if max_value == min_value:
                temp.append(0)
            else:
                temp.append((i - min_value) / (max_value - min_value))

        ## roc
        for i in range(0, self.repeat_count):
            if math.isnan(self.roc[i][0]):
                temp.append(0)
            else:
                temp.append(self.roc[i][0])
        
        ## dataX
        dataX = np.array([np.array(temp)])

        ## dataY
        dataY = self.model.predict(dataX)
        
        
        ## 开仓条件
        tLong = dataY[0][0]
#         tShort = dataY[0][1]
        if not self.position:
            fLong = (tLong > self.long_threshold) 
#             fShort = (tShort > self.short_threshold)
            if fLong:
                self.size = int(self.broker.cash / self.datas[0].close[0])
                self.order = self.buy(size=self.size)
                self.limitPrice = close_price + self.profit_target * close_price
                self.stopPrice = close_price - self.stop_target * close_price
#             elif fShort:
#                 self.order = self.sell(size=self.size)                
#                 self.limitPrice = close_price - self.profit_target * close_price
#                 self.stopPrice = close_price + self.stop_target * close_price

        ## 平仓逻辑
        if self.position:
            if self.position.size > 0:
                if close_price >= self.limitPrice or close_price <= self.stopPrice:
                    self.order = self.sell(size=self.size)
#             elif self.position.size < 0:
#                 if close_price <= self.limitPrice or close_price >= self.stopPrice:
#                     self.order = self.buy(size=self.size)
                    
    ## 日志记录
    def log(self, txt, dt=None, doprint=False):
        if self.params.printlog or doprint:
            dt = dt or self.datas[0].datetime.date(0)
            print(f'{dt.isoformat()},{txt}')

    # 记录交易执行情况（可选，默认不输出结果）
    def notify_order(self, order):
        # 如果 order 为 submitted/accepted，返回空
        if order.status in [order.Submitted, order.Accepted]:
            return
        # 如果 order 为 buy/sell executed，报告价格结果
        if order.status in [order.Completed]: 
            if order.isbuy():
                self.log(f'买入：\n价格：%.2f,\
                交易金额：-%.2f,\
                手续费：%.2f' % (order.executed.price, order.executed.value, order.executed.comm))
                self.buyprice = order.executed.price
                self.buycomm = order.executed.comm
            else:
                self.log(f'卖出:\n价格：%.2f,\
                交易金额：%.2f,\
                手续费：%.2f' % (order.executed.price, order.executed.price*self.size, order.executed.comm))
            self.bar_executed = len(self) 

        # 如果指令取消/交易失败, 报告结果
        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log('交易失败')
        self.order = None

    # 记录交易收益情况（可省略，默认不输出结果）
    def notify_trade(self,trade):
        if not trade.isclosed:
            return
        self.log(f'策略收益\n毛收益 {trade.pnl:.2f}, 净收益 {trade.pnlcomm:.2f}')

    # 回测结束后输出结果（可省略，默认输出结果）
    def stop(self):
        self.log('期末总资金 %.2f' %
                 (self.broker.getvalue()), doprint=True)

In [ ]:
from __future__ import (absolute_import, division, print_function,
                        unicode_literals)

import datetime
import os.path
import sys

import backtrader as bt

if __name__ == '__main__':
    # 创建 Cerebro 对象
    cerebro = bt.Cerebro()

    # 创建 Data Feed
    df_backtest.index = pd.to_datetime(df_backtest.index)
    start = df_backtest.index[0]
    end = df_backtest.index[-1]
    print(start, '-', end)
    data = bt.feeds.PandasData(dataname=df_backtest, fromdate=start, todate=end)
    
    # 将 Data Feed 添加至 Cerebro
    cerebro.adddata(data)

    # 添加策略 Cerebro
    cerebro.addstrategy(MyStrategy, 
                        model_path=model_path,
                        long_threshold=params["long_threshold"], 
                        short_threshold=params["short_threshold"], 
                        profit_target=params["profit_target"], 
                        stop_target=params["stop_target"],
                        repeat_count=params["repeat_count"], 
                        repeat_step=params["repeat_step"])
    
    # 设置初始资金
    cerebro.broker.setcash(100000.0)
    # 设置手续费为万二
    cerebro.broker.setcommission(commission=0.0002) 

    # 在开始时 print 初始账户价值
    print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())

    # 运行回测流程
    cerebro.run()

    # 在结束时 print 最终账户价值
    print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())

最后将回测结果进行可视化：

In [ ]:
import matplotlib
%matplotlib inline

# 画图并保存
fig = cerebro.plot(iplot=False)[0][0]
fig.set_size_inches(30, 18)
fig.savefig('{}/plot.png'.format(repository_name), dpi=100)

### 超参优化

您可能注意到回测的收益率并不高。同时，模型的 test accuracy 也非常低。这可能是因为糟糕的超参导致的。

因此，我们可以尝试通过超参优化提升模型的效果。

#### 上传数据到 S3

我们首先将实验数据集上传到 S3 指定目录：

In [ ]:
wr.s3.to_csv(df=df, path='s3://{}/{}/input/data_raw.csv'.format(bucket, repository_name))

#### 定义回测任务

首先我们先准备一个回测脚本：

In [ ]:
!mkdir {repository_name}/model

In [ ]:
%%writefile {repository_name}/model/train
#!/usr/bin/env python
from __future__ import print_function

import datetime
import json
import math
import numpy as np
import os
import pandas as pd
import sys
import talib as ta
import traceback

from keras.layers import Dropout, Dense
from keras.wrappers.scikit_learn import KerasClassifier
from keras.models import Sequential
from keras.wrappers.scikit_learn import KerasRegressor


# Optional
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# These are the paths to where SageMaker mounts interesting things in your container.
prefix = '/opt/ml/'

config_path = os.path.join(prefix, 'input/config')
input_path = os.path.join(prefix, 'input/data/training')
raw_path = os.path.join(input_path, 'data_raw.csv')
train_path = os.path.join(input_path, 'data_train.csv')
test_path = os.path.join(input_path, 'data_test.csv')
output_path = os.path.join(prefix, 'output')
model_path = os.path.join(prefix, 'model')


# Process and prepare the data
def get_data(params):
    
    # get and save hyperparameters
    long_threshold = float(params['long_threshold'])
    short_threshold = float(params['short_threshold'])
    profit_target = float(params['profit_target'])
    stop_target = float(params['stop_target'])

    repeat_count = int(params['repeat_count'])
    repeat_step = int(params['repeat_step'])
    look_back = repeat_count * repeat_step
    forward_window = int(params['forward_window'])
    
    # read data from s3
    df = pd.read_csv(raw_path, index_col=0)
    closePrice = df["close"]

    # use talib to calculate SMA and ROC
    ## set header for transformed data
    header = ["tradedate", "close"]
    for i in range(0, repeat_count):
        header.append("sma" + str((i+1) * repeat_step))
    for i in range(0, repeat_count):
        header.append("roc" + str((i+1) * repeat_step))
    header.append("long")
    header.append("short")

    data = []

    ## SMA
    inputs = {'close': np.array(closePrice)}
    sma = []
    for i in range(0, repeat_count):
        sma.append(ta.SMA(np.array(closePrice), timeperiod=(i+1) * repeat_step + 1))
    ## ROC - Rate of change : ((price/prevPrice)-1)*100
    roc = []
    for i in range(0, repeat_count):
        roc.append(ta.ROC(np.array(closePrice), timeperiod=(i+1) * repeat_step + 1))

    ## count long and short
    long_count = 0
    short_count = 0
    n_count = 0
    n = 0
    for idx in df.index:
        if n < len(df) - forward_window - 1:
            idx_0 = idx
            close_price = df.loc[idx, 'close']
            temp = []
            temp.append(idx)

            temp2 = []
            temp2.append(close_price)

            # sma
            for i in range(0, repeat_count):
                if np.isnan(sma[i][n]):
                    temp2.append(close_price)
                else:
                    temp2.append(sma[i][n])

            min_value = min(temp2)
            max_value = max(temp2)
            for i in temp2:
                if max_value == min_value:
                    temp.append(0)
                else:
                    temp.append((i - min_value) / (max_value - min_value))

            for i in range(0, repeat_count):
                if np.isnan(roc[i][n]):
                    temp.append(0)
                else:
                    temp.append(roc[i][n])

            rClose = closePrice[(n+1):min(len(df)-1, n+1+forward_window)].values.tolist()
            min_value = min(rClose)
            max_value = max(rClose)

            # long condition
            if max_value >= close_price * (1+profit_target) and min_value >= close_price * (1-stop_target):
                long_count += 1
                temp.append(1)
            else:
                temp.append(0)

            # short condition
            if min_value <= close_price * (1-stop_target) and max_value <= close_price * (1+profit_target):
                short_count += 1
                temp.append(1)
            else:
                temp.append(0)

            data.append(temp)
            n += 1

    print("long：%s, short：%s" % (long_count,short_count))
    df2 = pd.DataFrame(data, columns=header)
    df2.set_index('tradedate', inplace=True)
    print('Range:', df2.index[0], '-', df2.index[-1])
    
    # save data
    df_train = df2.iloc[:800]
    print('Training set:', df_train.index[0], '-', df_train.index[-1])
    print('Location:', train_path)
    df_train.to_csv(train_path)

    df_test = df2.iloc[800:1000]
    print('Testing set:', df_test.index[0], '-', df_test.index[-1])
    print('Location:', test_path)
    df_test.to_csv(test_path)


def data_process(df, yLen, b):
    
    dataX = []
    dataY = []
    for idx, row in df.iterrows():
        row1 = []
        r = row[1:len(row)-yLen]
        for a in r:
            row1.append(a)
        x = np.array(row1, dtype=float)
        y = np.array(row[len(row)-yLen:], dtype=float)
        b = len(x)
        dataX.append(x)
        dataY.append(y)
        
    dataX = np.array(dataX)
    dataY = np.array(dataY)
    
    return dataX, dataY, b


def build_classifier(b, yLen):
    
    print("build_classifier:b=%s,yLen=%s" % (b, yLen))
    model = Sequential()
    model.add(Dense(b, input_dim=b, kernel_initializer='normal', activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(int(b/2), kernel_initializer='normal', activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(yLen,kernel_initializer='normal', activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    return model


def generate_model(dataX, dataY, b, yLen):
    
    model = build_classifier(b, yLen)
    model.fit(dataX, dataY, epochs=100, batch_size=1)
    scores = model.evaluate(dataX, dataY, verbose=0)
    print("Training Data %s: %.2f%%" % (model.metrics_names[1], scores[1]*100))
    
    return model


def train():
    
    try:
        with open("{}/hyperparameters.json".format(config_path)) as json_file:
            params = json.load(json_file)
        print('Parameter load success')
    except Exception as e:
        print(e)

    get_data(params)
    
    print('Starting the training.')
    
    yLen = 2
    b = 0
    
    try:
        
        df = pd.read_csv(train_path)
        dataX, dataY, b = data_process(df, yLen, b)
        print('b:', b, 'yLen:', yLen)
        model = generate_model(dataX, dataY, b, yLen)
        model.save(os.path.join(model_path, 'model.h5'))
        
        print('Training is complete. Model saved.')
        
        df = pd.read_csv(test_path)
        dataX, dataY, b = data_process(df, yLen, b)
        print('b:', b, 'yLen:', yLen)
        scores = model.evaluate(dataX, dataY, verbose=0)
        print("Test Data %s: %.2f%%" % (model.metrics_names[1], scores[1]*100))
        
    except Exception as e:
        # Write out an error file. This will be returned as the failure
        # Reason in the DescribeTrainingJob result.
        trc = traceback.format_exc()
        with open(os.path.join(output_path, 'failure'), 'w') as s:
            s.write('Exception during training: ' + str(e) + '\n' + trc)
        # Printing this causes the exception to be in the training job logs
        print(
            'Exception during training: ' + str(e) + '\n' + trc,
            file=sys.stderr)
        # A non-zero exit code causes the training job to be marked as Failed.
        sys.exit(255)

        
if __name__ == '__main__':
    train()

    # A zero exit code causes the job to be marked a Succeeded.
    sys.exit(0)

#### 创建镜像仓库和容器镜像

接下来构建一个 python3 容器镜像和镜像仓库：

In [ ]:
ecs = boto3.client('ecs', region_name=aws_region)
ecr = boto3.client('ecr', region_name=aws_region)

In [ ]:
ecr.create_repository(repositoryName=repository_name)

In [ ]:
%%writefile {repository_name}/Dockerfile
FROM tensorflow/tensorflow:latest

RUN pip install --upgrade pip

RUN pip --no-cache-dir install \
        pandas \
        talib-binary

ENV PYTHONUNBUFFERED=TRUE
ENV PYTHONDONTWRITEBYTECODE=TRUE
ENV PATH="/opt/ml/code:${PATH}"

RUN mkdir -p /opt/ml/input/data
RUN mkdir -p /opt/ml/input/config
RUN mkdir -p /opt/ml/code
RUN mkdir -p /opt/ml/failure
RUN mkdir -p /opt/ml/model
RUN mkdir -p /opt/ml/output

COPY model /opt/ml/code
RUN chmod -R 777 /opt/ml/code
WORKDIR /opt/ml/code

In [ ]:
!docker build {repository_name} -t {repository_name}

In [ ]:
!docker tag {repository_name} {aws_account_id}.dkr.ecr.{aws_region}.amazonaws.com/{repository_name}

In [ ]:
!docker images

In [ ]:
!aws ecr get-login-password | docker login --username AWS --password-stdin {aws_account_id}.dkr.ecr.{aws_region}.amazonaws.com

In [ ]:
!docker push {aws_account_id}.dkr.ecr.{aws_region}.amazonaws.com/{repository_name}

In [ ]:
image_url = '{}.dkr.ecr.{}.amazonaws.com/{}'.format(aws_account_id, aws_region, repository_name)
print('镜像：', image_url)

#### 本地测试容器镜像

In [ ]:
!pip uninstall -y sagemaker
!pip install sagemaker==1.72.0

In [ ]:
params = { 
    "long_threshold" : 0.5,
    "short_threshold" : 1,
    "profit_target" : 0.02,
    "stop_target" : 0.01,
    "repeat_count": 20,
    "repeat_step": 1,
    "forward_window": 10
}

import sagemaker
from sagemaker.estimator import Estimator

role = sagemaker.get_execution_role()
session = sagemaker.Session()
estimator = Estimator(
    repository_name,
    role=role,
    train_instance_count=1,
    train_instance_type='local',
    output_path='s3://{}/{}/output'.format(bucket, repository_name),
    base_job_name=repository_name, 
    hyperparameters=params,
)

In [ ]:
estimator.fit('s3://{}/{}/input/data_raw.csv'.format(bucket, repository_name))

#### 运行超参优化任务

在开始之前，我们需要花一些时间来进行超参优化任务的配置。这些配置主要包括超参的定义和取值范围、目标参数的定义以及从日志中提取目标参数所需的正则表达式。这些配置信息通过 JSON 格式定义：

In [ ]:
from sagemaker.tuner import IntegerParameter, CategoricalParameter, ContinuousParameter, HyperparameterTuner

# 超参调优的参数配置
hyperparameter_ranges = {
    'long_threshold': ContinuousParameter(0.1, 0.9),
    'short_threshold': ContinuousParameter(1.0, 1.1),
    'profit_target': ContinuousParameter(0.01, 0.1),
    'stop_target': ContinuousParameter(0.01, 0.1),
    'repeat_count': IntegerParameter(10, 30),
    'repeat_step': IntegerParameter(1, 2),
    'forward_window': IntegerParameter(5, 30)
}

# 目标参数的配置
metric_definitions = [
    {
        'Name': 'loss', 
        'Regex': 'loss: ([0-9\\.]+)%'
    },
    {
        'Name': 'accuracy', 
        'Regex': 'accuracy: ([0-9\\.]+)%'
    },
    {
        'Name': 'train:accuracy', 
        'Regex': 'Training Data accuracy: ([0-9\\.]+)%'
    },
    {
        'Name': 'test:accuracy', 
        'Regex': 'Test Data accuracy: ([0-9\\.]+)%'
    }
]
objective_metric_name = "test:accuracy"

接下来，我们创建一个

In [ ]:
estimator = Estimator(
    image_url,
    role,
    train_instance_count=1,
    train_instance_type='ml.m5.4xlarge',
    output_path='s3://{}/{}/output'.format(bucket, repository_name),
    sagemaker_session=session
)

tuner = HyperparameterTuner(
    estimator, 
    objective_metric_name,
    hyperparameter_ranges,
    metric_definitions=metric_definitions,
    strategy='Bayesian',
    objective_type='Maximize', 
    max_jobs=500,
    max_parallel_jobs=10,
    base_tuning_job_name=repository_name,
    early_stopping_type='Auto'
)

tuner.fit('s3://{}/{}/input/data_raw.csv'.format(bucket, repository_name))

在超参优化的过程中，我们可以通过以下代码查看优化的结果,并选取一个理想的模型：

In [ ]:
tuner_analytics = tuner.analytics()
tuning_result = tuner_analytics.dataframe().sort_values(['FinalObjectiveValue'], ascending=False)

# select desired training result
i = 0
selected_training_job = tuning_result["TrainingJobName"].iloc[i]

params = { 
    "long_threshold" : float(tuning_result["long_threshold"].iloc[i]),
    "short_threshold" : float(tuning_result["short_threshold"].iloc[i]),
    "profit_target" : float(tuning_result["profit_target"].iloc[i]),
    "stop_target" : float(tuning_result["stop_target"].iloc[i]),
    "repeat_count": int(tuning_result["repeat_count"].iloc[i]),
    "repeat_step": int(tuning_result["repeat_step"].iloc[i]),
    "forward_window": int(tuning_result["forward_window"].iloc[i]),
}

import json
hyperparameter_path = '{}/hyperparameters.json'.format(repository_name)
print('超参路径：', hyperparameter_path)
with open(hyperparameter_path, 'w') as fp:
    json.dump(params, fp)

tuning_result.head(10)

训练完成后的模型由 SageMaker 自动打包并上传至 S3。我们只需找到最优的训练任务，并将其生产的 "model.h5" 的模型文件解压缩到实验目录下：

In [ ]:
import boto3
import io
import tarfile

s3 = boto3.client('s3')
s3_output_path = '{}/output/{}/output/model.tar.gz'.format(repository_name, selected_training_job)
print('s3路径：', s3_output_path)

bytestream = io.BytesIO(s3.get_object(Bucket=bucket, Key=s3_output_path)['Body'].read())
compressed_file = tarfile.open(fileobj=bytestream)
extract_path = '{}/'.format(repository_name)
compressed_file.extractall(path=extract_path)
model_path = extract_path + 'model.h5'
print('模型路径：', model_path)

模型和超参保存成功后，您可以再对这个策略进行回测，看策略的效果是否有所改善。